[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/708san/CV_data/blob/main/notebooks/colab/vision_smoke_test.ipynb)

# CVライブラリ：DETR・SAMのColab動作確認

このノートブックは、固定版のライブラリとモデルを使って物体検出・領域分割を試します。**Colabの「ランタイム」→「ランタイムのタイプを変更」からT4 GPUを選び、上から順に実行**してください。

初期設定では自分の画像をColabの一時ランタイムへアップロードします。画像をアップロードしたくない場合は、画像セルの`USE_OWN_IMAGE = False`にして合成画像を使えます。Notebookの画像はこのランタイム上で処理し、ライブラリから外部へ送信しません。モデル取得には数百MBの通信が発生します。結果は精度評価ではなく、APIの実行・戻り値・GPU利用のスモークテストです。

In [ ]:
import sys
print("Python:", sys.version)
if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError("このライブラリの対応Pythonは3.11〜3.13です。Colabのランタイムを確認してください。")

## 1. PyPI公開版をインストール

このNotebookはPyPIのcv-data-lesson 0.1.0a6とvision extraを使います。extraがPyTorch/Torchvisionも導入します。

この版ではHugging Face Hub 1.xを許容します。以前に別のモデル用パッケージを入れたランタイムは使わず、新しいColabランタイムで実行してください。

In [ ]:
%pip install 'cv-data-lesson[vision]==0.1.0a6'

## 2. ランタイムのGPUを確認

T4以外のGPUでも試せますが、その結果はT4の動作確認には数えません。GPUが見つからない場合はランタイム設定を変えて、最初から実行し直してください。

In [ ]:
import torch
from cvlesson import vision

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPUがありません。Colabのランタイム設定でT4 GPUを選んでください。")
device_name = torch.cuda.get_device_name(0)
print("GPU:", device_name)
if "T4" not in device_name:
    print("この実行はT4上の検証ではありません。GPU名を記録してください。")
print("cvlesson devices:", vision.available_devices())

## 3. 入力画像を選ぶ

初期設定では自分の画像を使います。Colab左側の「ファイル」欄から画像をアップロードし、次のセルの`IMAGE_PATH`に表示されたパスを設定してください。`files.upload()`のポップアップは使いません。画像はColabの一時ランタイム内で処理します。手元の画像を使わない場合は`USE_OWN_IMAGE = False`にします。

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

# 自分の画像を使わない場合はFalseにします。
USE_OWN_IMAGE = True

if USE_OWN_IMAGE:
    IMAGE_PATH = "/content/your-image.jpg"  # 左側「ファイル」欄からアップロードした画像のパス
    image_path = str(Path(IMAGE_PATH).resolve())
    if not Path(image_path).is_file():
        raise FileNotFoundError("Colab左側の「ファイル」欄から画像をアップロードし、IMAGE_PATHをそのファイル名に合わせてください。")
    with Image.open(image_path) as opened:
        image = opened.convert("RGB")
else:
    image = Image.new("RGB", (512, 512), "white")
    draw = ImageDraw.Draw(image)
    draw.rectangle((80, 90, 430, 420), fill=(225, 235, 245), outline=(20, 70, 120), width=8)
    draw.ellipse((170, 160, 340, 330), fill=(220, 70, 55), outline=(90, 20, 20), width=5)
    draw.rectangle((210, 345, 300, 390), fill=(40, 140, 70))
    image_path = "/content/cvlesson_smoke.png"
    image.save(image_path)

print("画像サイズ:", image.size)
display(image)

## 4. DETRで物体検出

初回は固定版のモデル重み（約167 MB）を取得します。COCOの80カテゴリの検出モデルなので、図形画像で検出結果が0件でも、それだけでAPIの失敗とは判定しません。

In [ ]:
detector = vision.load_detector(device="cuda:0")
detections = detector.predict(image, threshold=0.05)
print("検出件数:", len(detections))
for item in detections:
    x1, y1, x2, y2 = item.box
    assert item.source_size == image.size
    assert 0 <= x1 < x2 <= image.width
    assert 0 <= y1 < y2 <= image.height
    assert 0.0 <= item.score <= 1.0
    print(item.label, round(item.score, 3), item.box)
print("DETR推論と戻り値の確認: OK")

## 5. SAMで枠からマスクを作る

初回はsafetensors形式の重み（約375 MB）を取得します。指定枠に対する候補マスクとスコアが返り、対象名は返りません。

SAMのスコアはマスク品質の予測値です。実際のIoUや正解確率ではなく、0〜1の範囲も保証されないため、値の範囲ではなく有限数かどうかを確認します。

In [ ]:
segmenter = vision.load_segmenter(device="cuda:0")
# SAMには画像座標の矩形を渡します。画像サイズに合わせ、中央付近を指定します。
width, height = image.size
box = (0.1 * width, 0.1 * height, 0.9 * width, 0.9 * height)
segmentation = segmenter.predict(image_path, box=box)
assert segmentation.mask.shape == (image.height, image.width)
assert str(segmentation.mask.dtype) == "bool"
from math import isfinite
assert isfinite(segmentation.score)
print("SAMの予測品質値（実IoU・確率ではありません）:", segmentation.score)
print("マスク形状:", segmentation.mask.shape)
print("マスク画素数:", int(segmentation.mask.sum()))
print("SAM推論と戻り値の確認: OK")

In [ ]:
import numpy as np

overlay = np.asarray(image).copy()
overlay[segmentation.mask] = (
    0.55 * overlay[segmentation.mask] + 0.45 * np.array([40, 180, 80])
).astype(np.uint8)
display(Image.fromarray(overlay))

## 6. 検出枠をSAMに渡す（任意）

DETRの検出があった場合、その先頭の枠をSAMへ渡します。検出ラベルはDETR、マスクはSAMの出力です。

In [ ]:
if detections:
    combined = segmenter.segment_detections(image, detections[:1])
    assert combined[0].segmentation.mask.shape == (image.height, image.width)
    print(combined[0].detection.label, combined[0].segmentation.mask.shape)
    print("検出枠からSAMへの連携: OK")
else:
    print("この図形画像ではDETRの検出が0件でした。枠を指定したSAMテストは完了しています。")

## 結果の記録

- Python / PyTorchの版：上の出力
- GPU名：上の出力
- DETR：エラーなく推論し、結果型・座標を確認できたか
- SAM：元画像と同じ大きさのboolマスクが返ったか
- エラーが出た場合：最初に失敗したセルと全文を開発者へ共有

ここで確認したのは推論処理が動くことです。実データの精度や課題への適合を評価したものではありません。

## 7. 実データの画像と正解マスクを比べる

FruitSeg30のtest splitから1組を取得します。CC BY 4.0の出典表示が必要です。SAMには正解マスクから求めた矩形をpromptとして渡します。これはモデル単体の領域提案を試す比較例で、実利用時に正解マスクを渡す手順ではありません。1画像のIoUは動作確認用で、モデル全体の精度を示しません。

In [ ]:
from cvlesson import datasets

print(datasets.info("fruitseg30"))  # CC BY 4.0・出典と変更の表示が必要
fruit_data = datasets.get("fruitseg30")
fruit_frame = fruit_data.to_df()
sample = fruit_frame[fruit_frame["split"] == "test"].iloc[0]
with Image.open(sample["image_path"]) as opened:
    fruit_image = opened.convert("RGB")
with Image.open(sample["mask_path"]) as opened:
    target_mask = np.asarray(opened.convert("L")) > 127
ys, xs = np.where(target_mask)
if len(xs) == 0:
    raise RuntimeError("選んだ正解マスクに前景画素がありません")
target_box = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
real_detections = detector.predict(sample["image_path"], threshold=0.05)
real_segmentation = segmenter.predict(sample["image_path"], box=target_box)
predicted_mask = real_segmentation.mask
intersection = int(np.logical_and(target_mask, predicted_mask).sum())
union = int(np.logical_or(target_mask, predicted_mask).sum())
iou = intersection / union if union else 1.0
print("データ:", sample["label"], " / test画像:", sample["sample_id"])
print("DETR推論件数:", len(real_detections))
print("SAMと正解マスクのIoU（この1画像のみ）:", round(iou, 3))

ground_truth_view = np.asarray(fruit_image).copy()
ground_truth_view[target_mask] = (0.55 * ground_truth_view[target_mask] + 0.45 * np.array([40, 180, 80])).astype(np.uint8)
prediction_view = np.asarray(fruit_image).copy()
prediction_view[predicted_mask] = (0.55 * prediction_view[predicted_mask] + 0.45 * np.array([220, 80, 50])).astype(np.uint8)
print("正解マスク（緑）:")
display(Image.fromarray(ground_truth_view))
print("SAM予測（赤）:")
display(Image.fromarray(prediction_view))